# Full-tensor mode sources and monitors

This companion to [Mode Sources and Monitors](modal_sources_monitors.ipynb) follows the same sequence: a center-frequency source, a seven-profile broadband source, and a width-step junction. The dielectric core has **nonzero off-diagonal permittivity**, and `farjadpour_full` retains the coupled electric-support and shared-node coefficients.

We use the validated lossless 3D, rectilinear-grid, CPU JAX path. The synthetic core is 0.6 × 0.6 µm in air, with

$$\epsilon_r=\begin{pmatrix}6&0.2&0.1\\0.2&5&0.3\\0.1&0.3&4\end{pmatrix}.$$

These are mixed-polarization guided modes: `polarization=None` is intentional. Mode solving, source injection, and `results.mode()` use the discrete tensor operator. Transverse cell widths vary by about 3:1. The grid and material remain invariant along propagation near each port; strict TE/TM selection is outside this mixed-polarization workflow. This notebook uses CPU JAX; GPU JAX and CUDA are also covered by the library validation suite.

**What to look for:** nonzero constitutive cross terms, mixed eigenfields, small unwanted guided power in the straight guide, agreement between sampled modal flux and independently measured flux, and scattering at the width step. The final cells contain numerical assertions and a detached-result save/load check. This is an illustrative device experiment; the broader analytical and independent MPB evidence is in the [validation report](../../benchmarks/reports/issue361/README.md).

## Setup

Use the checkout containing issue #361's tensor implementation. Install BeamZ and the notebook dependencies in its environment (`pip install -e '.[dev]'`). The saved notebook contains the full run; **Restart Kernel and Run All** reproduces it. `BEAMZ_DOCS_TEST=1` reduces only the spectral samples and profile count, while retaining the full propagation time.

In [ ]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

from dataclasses import replace
from pathlib import Path
from tempfile import TemporaryDirectory
import platform
import time

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
import beamz as bz
from beamz.design.discretization import MaterialGrid
from beamz.design.raster import Box, Grid, Material, Object, RasterOptions, Scene, rasterize

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (10, 4), "figure.dpi": 110,
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "lines.linewidth": 1.8,
})
print(f"Python {platform.python_version()} | JAX {jax.__version__} | {jax.devices()}")
print("BeamZ:", Path(bz.__file__).resolve())
test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"
pd.options.display.float_format = lambda value: f"{value:.5g}"

## Geometry, frequency band, and grid

Coordinates run from zero to the domain extent. The longitudinal cell size is 100 nm; a sinusoidal transverse mapping concentrates cells around the core, with widths of about 50–150 nm. Set `rectilinear=False` to recover the cubic-grid example. The core tensor is real, symmetric, and positive definite. CPML is 1.2 µm thick at the longitudinal ends and 0.6 µm thick transversely. Source and monitor apertures exclude CPML and include the guided tails.

The 1.55 µm center wavelength and nine monitor frequencies span ±10% in frequency. Seven Chebyshev profiles cover ±1.5 pulse widths. The acquisition lasts 20 inverse bandwidths (about 1.29 ps), including the longer tails introduced by broadband profile reconstruction. Decreasing `spacing` is the next accuracy check; it increases both the grid size and number of time steps.

In [ ]:
um = bz.um
wavelength0 = 1.55 * um
freq0 = bz.LIGHT_SPEED / wavelength0
fwidth = 0.08 * freq0
freqs = freq0 * np.linspace(0.9, 1.1, 3 if test_mode else 9)
wavelengths_um = bz.LIGHT_SPEED / freqs / um
center_index = len(freqs) // 2
profile_count = 3 if test_mode else 7
spacing = 0.1 * um
domain = np.array([6.4, 3.6, 3.6]) * um
shape = tuple(np.rint(domain / spacing).astype(int))
np.testing.assert_allclose(np.array(shape) * spacing, domain)
core_epsilon = np.array([[6.0, 0.2, 0.1], [0.2, 5.0, 0.3], [0.1, 0.3, 4.0]])
assert np.linalg.eigvalsh(core_epsilon).min() > 0
rectilinear = True
edges = [np.linspace(0, length, count + 1)
         for length, count in zip(domain, shape)]
if rectilinear:
    for axis in (1, 2):
        u = np.linspace(0, 1, shape[axis] + 1)
        edges[axis] = domain[axis] * (u + 0.08 * np.sin(2 * np.pi * u))
raster_grid = Grid(*edges)
display(pd.DataFrame({
    "Axis": list("xyz"),
    "Minimum cell width (nm)": [np.diff(e).min() / 1e-9 for e in edges],
    "Maximum cell width (nm)": [np.diff(e).max() / 1e-9 for e in edges],
}))
raster_options = RasterOptions(smoothing="farjadpour_full")
materials = (Material(), Material(epsilon_r=tuple(map(tuple, core_epsilon))))
core = Object(Box((0, 1.5*um, 1.5*um), (domain[0], 2.1*um, 2.1*um)), 1)


def material_grid(*, junction=False):
    objects = (core,)
    if junction:
        # Widen the output guide from 0.6 to 1.0 um after x=3.2 um.
        objects += (Object(Box((3.2*um, 1.3*um, 1.5*um),
                               (domain[0], 2.3*um, 2.1*um)), 1),)
    return MaterialGrid.from_raster_result(
        rasterize(Scene(materials, objects), raster_grid, options=raster_options)
    )

straight_material = material_grid()
junction_material = material_grid(junction=True)
assert straight_material.uses_full_permittivity
cross_term_maxima = {
    name: float(np.max(np.abs(values[3:])))
    for name, values in straight_material.yee_tensors.items()
}
assert all(value > 0 for value in cross_term_maxima.values())
display(pd.Series(cross_term_maxima, name="Maximum absolute permittivity cross term"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7), layout="constrained")
z_edges, y_edges, x_edges = (straight_material.grid.axis_edges(a) / um for a in "zyx")
x_index = int(2.0 * um / spacing)
z_index = int(1.8 * um / spacing)
for ax, component, title in zip(axes[:2], (1, 3), (r"$\epsilon_{yy}$", r"$\epsilon_{xy}$")):
    values = straight_material.tensors["epsilon"][component, :, :, x_index]
    artist = ax.pcolormesh(y_edges, z_edges, values, shading="flat", cmap="viridis")
    fig.colorbar(artist, ax=ax, label="Relative permittivity")
    ax.set(xlabel="y (µm)", ylabel="z (µm)", title=title, aspect="equal")
artist = axes[2].pcolormesh(x_edges, y_edges,
    junction_material.tensors["epsilon"][1, z_index], shading="flat", cmap="viridis")
axes[2].axvline(2.0, color="white", linestyle="--", label="Source")
axes[2].axvline(2.6, color="#E69F00", linestyle=":", label="Input monitor")
axes[2].axvline(4.4, color="#56B4E9", linestyle="-.", label="Output monitor")
axes[2].set(xlabel="x (µm)", ylabel="y (µm)", title="Width step at z = 1.8 µm", aspect="equal")
axes[2].legend(fontsize=8, loc="upper right")
plt.show()

## Source and monitors

Both modes are requested without a TE/TM filter. The source launches mode 0 along +x. Mode monitors at x=2.6 and 4.4 µm share the same aperture as independent flux monitors. A field monitor records `Ey` in the mid-height plane.

The input and output mode bases are solved from their own local material snapshots. Their full constitutive neighborhoods remain longitudinally invariant, including in the width-step device.

In [ ]:
mode_spec = bz.ModeSpec(num_modes=2, mode_index=0, polarization=None, target_neff=2.0)
plane_size = (0, 2.2*um, 2.2*um)
pulse = bz.GaussianPulse(freq0=freq0, fwidth=fwidth, offset=4.0)
source_single = bz.ModeSource(
    center=(2.0*um, 1.8*um, 1.8*um), size=plane_size, direction="+",
    source_time=pulse, mode_spec=replace(mode_spec, num_freqs=1),
)
source_broadband = source_single.updated_copy(
    mode_spec=replace(mode_spec, num_freqs=profile_count)
)
monitors = []
for name, x in (("input", 2.6), ("output", 4.4)):
    monitors.extend([
        bz.ModeMonitor(center=(x*um, 1.8*um, 1.8*um), size=plane_size,
                       freqs=freqs, mode_spec=mode_spec, name=name),
        bz.FluxMonitor(center=(x*um, 1.8*um, 1.8*um), size=plane_size,
                       freqs=freqs, name=name + "_flux"),
    ])
monitors.append(bz.FieldMonitor(
    center=(domain[0]/2, domain[1]/2, 1.8*um),
    size=(domain[0], domain[1], 0), freqs=[freq0], fields=("Ey",), name="field",
))
boundaries = [
    bz.PML(edges=["left", "right"], thickness=1.2*um, formulation="cpml"),
    bz.PML(edges=["bottom", "top", "front", "back"], thickness=0.6*um, formulation="cpml"),
]
sim_single = bz.Simulation(
    material_grid=straight_material, sources=[source_single], monitors=monitors,
    boundaries=boundaries, run_time=20/fwidth, setup_device="cpu",
)
sim_broadband = sim_single.updated_copy(sources=(source_broadband,))
sim_junction = sim_broadband.updated_copy(material_grid=junction_material)
print(f"Grid {shape}, dt={sim_single.dt*1e18:.2f} as, {sim_single.num_steps} steps")
print("Broadband profile wavelengths (µm):",
      np.round(bz.LIGHT_SPEED / source_broadband.profile_frequencies() / um, 4))

## Inspect the broadband interpolation

These are the partition weights used by the source compiler. They sum to one and select exactly one profile at each Chebyshev node. Negative weights between nodes are expected for polynomial interpolation. The sampled monitor band lies inside the profile band; outside it, the compiler holds the nearest end profile.

In [ ]:
from beamz.devices.sources.time import partition_weights_by_frequency

profile_frequencies = source_broadband.profile_frequencies()
plot_frequencies = np.linspace(profile_frequencies[0], profile_frequencies[-1], 201)
weights = partition_weights_by_frequency(plot_frequencies, profile_frequencies)
np.testing.assert_allclose(weights.sum(axis=0), 1, atol=1e-12)
np.testing.assert_allclose(
    partition_weights_by_frequency(profile_frequencies, profile_frequencies),
    np.eye(profile_count), atol=1e-12,
)
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
for index, values in enumerate(weights):
    ax.plot(plot_frequencies/freq0, values, label=f"Profile {index}",
            linestyle="-" if index % 2 == 0 else "--")
ax.axhline(0, color="0.5", linewidth=0.7)
ax.set(xlabel="Frequency / center frequency", ylabel="Interpolation weight",
       title=f"{profile_count} Chebyshev profiles")
ax.legend(fontsize=9, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.show()

## Solve and inspect the tensor eigenmodes

`solve_modes()` previews the same discrete modes used for launch. The fields below are collocated for display; their Yee-supported values determine the residual checks. Each row uses one common electric-field scale, so relative component strengths are visible. The three component magnitudes demonstrate why a strict TE/TM label is inappropriate.

Candidates use the solver's effective-index ordering near `target_neff`. This example has separated modes; arbitrary frequency sweeps through degeneracies require additional field-overlap tracking.

In [ ]:
modes = source_single.solve_modes(sim_single, freqs=[freq0])
from beamz.analysis import mode_data_to_dataframe
display(mode_data_to_dataframe(modes))
display(pd.DataFrame([dict(d) for d in modes.diagnostics[0]["modes"]])[[
    "full_tensor_operator", "tensor_faraday_residual", "tensor_ampere_residual",
    "tensor_iterations",
]])
assert all(d["full_tensor_operator"] for d in modes.diagnostics[0]["modes"])

fig, axes = plt.subplots(2, 3, figsize=(10, 6), layout="constrained")
vertical_edges, horizontal_edges = (np.asarray(v)/um for v in modes.grid_edges)
for mode_index, row in enumerate(axes):
    fields = np.abs(modes.e_fields[0, mode_index])
    scale = np.max(fields)
    for component, ax in enumerate(row):
        artist = ax.pcolormesh(horizontal_edges, vertical_edges,
            fields[component]/scale, vmin=0, vmax=1, cmap="magma", shading="flat")
        ax.set(xlabel="y (µm)", ylabel="z (µm)", aspect="equal",
               title=f"Mode {mode_index}: |E{'xyz'[component]}|")
fig.colorbar(artist, ax=axes.ravel().tolist(), label="Relative amplitude (one scale per mode)")
plt.show()

## Run the single-profile and broadband straight guides

Both runs use the same pulse, grid, monitors, and propagation time. Only the number of frequency-dependent source profiles changes. Results are source-spectrum normalized by BeamZ; no measured spectrum is rescaled to force a flat unit-power curve.

In [ ]:
started = time.perf_counter()
result_single = sim_single.run(backend="jax", progress=False)
result_broadband = sim_broadband.run(backend="jax", progress=False)
print(f"Two straight-guide simulations: {time.perf_counter()-started:.1f} s")
modal_single = result_single.mode("output")
modal_broadband = result_broadband.mode("output")
input_broadband = result_broadband.mode("input")

In [ ]:
result_broadband.plot_field("field", "Ey", frequency=freq0, val="real",
    cmap="RdBu_r", show_grid=False)
plt.show()

## Broadband launch purity and power accounting

`abs(amps)**2` reports each mode's discrete Poynting-flux contribution. Each directional basis includes the monitor's Yee interpolation before it is normalized to one watt. Source calibration and independent flux monitors use the same convention. `modal_flux` additionally includes interference cross terms and is the quantity to compare with `FluxMonitor.flux` for a general superposition.

The unwanted-guided-power ratio adds all backward power and forward mode-1 power, divided by forward mode-0 power. It does not include radiation outside this two-mode basis. The raw pulse plot is divided by its own central-frequency value only to show the pulse envelope, as labeled.

In [ ]:
def powers(mode_data):
    return (np.abs(mode_data.amps.sel(direction="+").values)**2,
            np.abs(mode_data.amps.sel(direction="-").values)**2)


def unwanted_ratio(mode_data):
    forward, backward = powers(mode_data)
    return (backward.sum(axis=1) + forward[:, 1:].sum(axis=1)) / forward[:, 0]


forward_single, backward_single = powers(modal_single)
forward_broadband, backward_broadband = powers(modal_broadband)
flux_single = np.asarray(result_single["output_flux"].flux)
flux_broadband = np.asarray(result_broadband["output_flux"].flux)
order = np.argsort(wavelengths_um)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for label, data, flux, linestyle in (
    ("Single profile", modal_single, flux_single, "--"),
    ("Broadband", modal_broadband, flux_broadband, "-"),
):
    axes[0].plot(wavelengths_um[order], flux[order], linestyle, label=label + " flux")
    axes[0].plot(wavelengths_um[order], np.asarray(data.modal_flux)[order],
                 "o", markersize=4, fillstyle="none", label=label + " modal flux")
    axes[1].semilogy(wavelengths_um[order], unwanted_ratio(data)[order],
                     linestyle + "o", markersize=4, label=label)
raw_flux = np.asarray(result_single.renormalize(None)["output_flux"].flux)
axes[2].plot(wavelengths_um[order], (raw_flux/raw_flux[center_index])[order], "o-")
for ax in axes:
    ax.set_xlabel("Wavelength (µm)")
    ax.grid(alpha=0.2)
axes[0].set(ylabel="Source-normalized sampled power (W)", title="Independent flux agreement")
axes[1].set(ylabel="Unwanted / forward-mode-0 power", title="Guided launch impurity")
axes[2].set(ylabel="Raw flux / central-frequency raw flux", title="Pulse-weighted spectrum")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=9)
plt.show()

rows = []
for label, data, flux in (("Single profile", modal_single, flux_single),
                          ("Broadband", modal_broadband, flux_broadband)):
    forward, backward = powers(data)
    rows.append({"Source": label,
        "Center P0 (W)": forward[center_index, 0],
        "Center flux (W)": flux[center_index],
        "Center backward fraction": backward[center_index].sum()/forward[center_index, 0],
        "Max guided impurity": unwanted_ratio(data).max(),
        "Max flux discrepancy": np.max(np.abs(np.asarray(data.modal_flux)-flux)/np.abs(flux)),
    })
straight_checks = pd.DataFrame(rows).set_index("Source")
display(straight_checks)

## Width-step junction

The same tensor core widens from 0.6 to 1.0 µm at x=3.2 µm. We reuse the broadband source, while the output mode monitor solves the wider guide's own tensor modes. The input monitor resolves the reflected guided field.

Transmission and reflection below are divided by the matching straight run's incident **mode-0 flux contribution** at the input plane. They are two-mode guided contributions. Their deficit also contains radiation, omitted guided modes, finite-aperture error, and discretization; it should not be interpreted as absorption in this lossless material.

In [ ]:
started = time.perf_counter()
result_junction = sim_junction.run(backend="jax", progress=False)
junction_input = result_junction.mode("input")
junction_output = result_junction.mode("output")
print(f"Width-step simulation and projection: {time.perf_counter()-started:.1f} s")
result_junction.plot_field("field", "Ey", frequency=freq0, val="real",
    cmap="RdBu_r", show_grid=False)
plt.show()

In [ ]:
incident = powers(input_broadband)[0][:, 0]
transmitted = powers(junction_output)[0] / incident[:, None]
reflected = powers(junction_input)[1] / incident[:, None]
guided_total = transmitted.sum(axis=1) + reflected.sum(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for index in range(mode_spec.num_modes):
    axes[0].plot(wavelengths_um[order], transmitted[order, index], "o-", label=f"T mode {index}")
    axes[1].plot(wavelengths_um[order], reflected[order, index], "o-", label=f"R mode {index}")
axes[0].plot(wavelengths_um[order], guided_total[order], "k--", label="Sum of guided T + R")
axes[0].axhline(1, color="0.5", linewidth=0.8)
for ax in axes:
    ax.set(xlabel="Wavelength (µm)", ylabel="Fraction of reference incident modal power")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
axes[0].set_title("Wider-guide transmission")
axes[1].set_title("Input-guide reflection")
plt.show()
display(pd.DataFrame({
    "Wavelength (µm)": wavelengths_um,
    "Guided T": transmitted.sum(axis=1), "Guided R": reflected.sum(axis=1),
    "1 - guided T - guided R": 1-guided_total,
}).iloc[order].reset_index(drop=True))
max_excess = max(0.0, float(guided_total.max()-1))
display(Markdown(
    f"At 1.55 µm, the two-mode guided transmission is **{100*transmitted[center_index].sum():.3f}%** "
    f"and reflection is **{100*reflected[center_index].sum():.4f}%**. "
    f"Across this band, the largest excess of guided T + R over unity is **{100*max_excess:.3f}%**. "
    "This coarse-grid accounting residual needs grid, aperture, and monitor-position refinement; "
    "a negative value in the last column does not represent material gain."
))


## Checks and detached-result round trip

These assertions check center-frequency launch purity, sampled power accounting, power transport between straight-guide planes, finite junction amplitudes, and persistence of complex modal amplitudes. They use fixed tolerances and do not renormalize errors away. Broadband off-center errors and the junction power deficit remain visible above.

The archive contains the timestep, source specifications, monitor fields, and full Yee material context. We reload it without accessing the live simulation and repeat the public modal extraction. A temporary directory avoids leaving a large result file beside the notebook; replace it with a permanent path to keep the archive.

In [ ]:
assert np.isfinite(modes.neffs).all()
assert unwanted_ratio(modal_single)[center_index] < 1e-5
for data, flux in ((modal_single, flux_single), (modal_broadband, flux_broadband)):
    np.testing.assert_allclose(data.flux, flux, rtol=1e-6, atol=1e-8)
    assert np.max(np.abs(np.asarray(data.modal_flux)-flux)/np.abs(flux)) < 2e-3
np.testing.assert_allclose(forward_broadband[center_index, 0],
                          incident[center_index], rtol=2e-3)
assert np.isfinite(transmitted).all() and np.isfinite(reflected).all()
with TemporaryDirectory(prefix="beamz-tensor-notebook-") as folder:
    archive_path = Path(folder) / "straight-tensor.npz"
    result_broadband.save(archive_path)
    restored = bz.SimulationResults.load(archive_path)
    np.testing.assert_allclose(restored.mode("output").amps, modal_broadband.amps,
                               rtol=1e-12, atol=1e-12)
print("All notebook checks passed, including complex-amplitude save/load at 1e-12.")
assert unwanted_ratio(modal_broadband).max() < unwanted_ratio(modal_single).max()
display(Markdown(
    f"The worst unwanted guided-power fraction falls from "
    f"**{unwanted_ratio(modal_single).max():.2e}** with one profile to "
    f"**{unwanted_ratio(modal_broadband).max():.2e}** with {profile_count} profiles. "
    f"The broadband sampled-flux discrepancy is at most "
    f"**{np.max(np.abs(np.asarray(modal_broadband.modal_flux)-flux_broadband)/np.abs(flux_broadband)):.2e}**."
))

# Protect absolute flux and modal watts together at the central frequency.
for data, flux in ((modal_single, flux_single), (modal_broadband, flux_broadband)):
    np.testing.assert_allclose(flux[center_index], 1.0, rtol=2e-3)
    np.testing.assert_allclose(powers(data)[0][center_index, 0], flux[center_index], rtol=2e-3)


## Next steps

Refine the rectilinear grid and enlarge the aperture before relying on a device's absolute S-parameters. For larger width steps, include additional confined output modes and check the unresolved power budget. Inspect field overlaps when sweeping through a mode crossing.

See the [tensor workflow guide](../../docs/tensor-modal-workflows.md) for supported combinations, phasor and power conventions, and archive restrictions, and the [validation report](../../benchmarks/reports/issue361/README.md) for analytical Yee checks, six propagation directions, reciprocity, second-order convergence, and an independent Meep/MPB reference.